In [120]:
import cv2
import numpy as np
import joblib

import os
import urllib.request
from collections import deque
from mediapipe import Image, ImageFormat
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [121]:
model = joblib.load('posture_model_optuna.pkl')
scaler = joblib.load('scaler_optuna.pkl')

In [122]:
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
MODEL_PATH = "pose_landmarker_lite.task"
if not os.path.exists(MODEL_PATH):
    print("📥 Downloading pose model (~10 MB)...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("✅ Downloaded")


In [123]:
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False
)
detector = vision.PoseLandmarker.create_from_options(options)

In [124]:
# ── Constante landmarks ──────────────────────────────────────────────────────
NOSE           = 0
LEFT_SHOULDER  = 11
RIGHT_SHOULDER = 12
LEFT_ELBOW     = 13
RIGHT_ELBOW    = 14

LANDMARK_INDICES = [NOSE, LEFT_SHOULDER, RIGHT_SHOULDER, LEFT_ELBOW, RIGHT_ELBOW]

# ── Helper ───────────────────────────────────────────────────────────────────
def get_pt(landmarks, idx):
    lm = landmarks[idx]
    return np.array([lm.x, lm.y, lm.z])

# ── Feature extraction ───────────────────────────────────────────────────────
def compute_features(landmarks):
    nose           = get_pt(landmarks, NOSE)
    left_shoulder  = get_pt(landmarks, LEFT_SHOULDER)
    right_shoulder = get_pt(landmarks, RIGHT_SHOULDER)
    left_elbow     = get_pt(landmarks, LEFT_ELBOW)
    right_elbow    = get_pt(landmarks, RIGHT_ELBOW)

    shoulder_mid   = (left_shoulder + right_shoulder) / 2.0
    shoulder_width = np.linalg.norm(left_shoulder - right_shoulder)
    if shoulder_width < 1e-6:
        shoulder_width = 1.0

    f = []

    # 1. Head tilt
    vec      = nose - shoulder_mid
    vertical = np.array([0, 1, 0])
    cos_angle = np.dot(vec, vertical) / (np.linalg.norm(vec) * np.linalg.norm(vertical) + 1e-8)
    f.append(np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0))))

    # 2. Shoulder slope
    shoulder_vec = right_shoulder - left_shoulder
    horizontal   = np.array([1, 0, 0])
    cos_slope = np.dot(shoulder_vec, horizontal) / (np.linalg.norm(shoulder_vec) * np.linalg.norm(horizontal) + 1e-8)
    f.append(np.degrees(np.arccos(np.clip(cos_slope, -1.0, 1.0))))

    # 3. Nose distance from shoulder midpoint
    f.append(np.linalg.norm(nose - shoulder_mid) / shoulder_width)

    # 4-7. Left elbow relative to left shoulder
    f.append((left_elbow[2]  - left_shoulder[2])  / shoulder_width)
    f.append((left_elbow[0]  - left_shoulder[0])  / shoulder_width)
    f.append((left_elbow[1]  - left_shoulder[1])  / shoulder_width)
    f.append(np.linalg.norm(left_elbow - left_shoulder) / shoulder_width)

    # 8-11. Right elbow relative to right shoulder
    f.append((right_elbow[2] - right_shoulder[2]) / shoulder_width)
    f.append((right_elbow[0] - right_shoulder[0]) / shoulder_width)
    f.append((right_elbow[1] - right_shoulder[1]) / shoulder_width)
    f.append(np.linalg.norm(right_elbow - right_shoulder) / shoulder_width)

    # 12. Shoulder asymmetry (y-axis)
    f.append((left_shoulder[1] - right_shoulder[1]) / shoulder_width)

    # 13. Head rotation
    vec_proj      = vec.copy();          vec_proj[1] = 0
    shoulder_proj = shoulder_vec.copy(); shoulder_proj[1] = 0
    if np.linalg.norm(vec_proj) > 0 and np.linalg.norm(shoulder_proj) > 0:
        cos_rot = np.dot(vec_proj, shoulder_proj) / (np.linalg.norm(vec_proj) * np.linalg.norm(shoulder_proj) + 1e-8)
        f.append(np.degrees(np.arccos(np.clip(cos_rot, -1.0, 1.0))))
    else:
        f.append(0.0)

    # 14. Elbow-elbow distance
    f.append(np.linalg.norm(left_elbow - right_elbow) / shoulder_width)

    return np.array(f).reshape(1, -1)

# ── Model + class order ──────────────────────────────────────────────────────
model  = joblib.load('posture_model_optuna.pkl')
scaler = joblib.load('scaler_optuna.pkl')

assert list(model.classes_) == [0, 1], f"Unexpected class order: {model.classes_}"
GOOD_CLASS_IDX = list(model.classes_).index(0)

In [125]:
alpha = 0.7
prev_landmarks = None

def smooth_landmarks(landmarks):
    global prev_landmarks

    if prev_landmarks is None:
        prev_landmarks = landmarks
        return landmarks

    smoothed = []

    for i in range(len(landmarks)):
        x = alpha * prev_landmarks[i].x + (1 - alpha) * landmarks[i].x
        y = alpha * prev_landmarks[i].y + (1 - alpha) * landmarks[i].y
        z = alpha * prev_landmarks[i].z + (1 - alpha) * landmarks[i].z

        smoothed.append(type(landmarks[i])(x, y, z))

    prev_landmarks = smoothed
    return smoothed

In [126]:
from collections import deque

feat_buffer = deque(maxlen=10)

In [127]:
import time

CONNECTIONS = [
    (0,1), (0,4), (1,2), (2,3), (4,5), (5,6), (7,0), (8,0),
    (9,0), (10,0), (11,12), (11,13), (12,14)
]

cap = cv2.VideoCapture(0)

BUFFER_SIZE    = 45        # 1.5s at 30fps
THRESHOLD      = 0.50
CALIB_SECONDS  = 5

prob_buffer   = deque(maxlen=BUFFER_SIZE)
calib_frames  = []
calib_offset  = None       # shape (1, 14) — personal correction
calibrating   = True
calib_start   = time.time()

print("CALIBRATION: sit in good posture for 15 seconds...")

# reinitialize detector fresh
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False
)
detector = vision.PoseLandmarker.create_from_options(options)
model  = joblib.load('posture_model_optuna.pkl')
scaler = joblib.load('scaler_optuna.pkl')
GOOD_CLASS_IDX = list(model.classes_).index(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = Image(image_format=ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_img)

    # ── CALIBRATION PHASE ────────────────────────────────────────────────────
    if calibrating:
        elapsed   = time.time() - calib_start
        remaining = max(0, int(CALIB_SECONDS - elapsed))

        if result.pose_landmarks:
            landmarks = result.pose_landmarks[0]

            landmarks = smooth_landmarks(landmarks)
            
            feat = compute_features(landmarks)            # raw unscaled (1,14)
            calib_frames.append(feat[0])

        # overlay
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (frame.shape[1], frame.shape[0]), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.45, frame, 0.55, 0, frame)

        h, w = frame.shape[:2]
        cv2.putText(frame, "CALIBRATING", (w//2 - 160, h//2 - 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 220, 80), 3, cv2.LINE_AA)
        cv2.putText(frame, "Sit in your best posture", (w//2 - 175, h//2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2, cv2.LINE_AA)
        cv2.putText(frame, f"{remaining}s", (w//2 - 35, h//2 + 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 2.2, (0, 220, 80), 4, cv2.LINE_AA)

        if elapsed >= CALIB_SECONDS:
            calibrating = False
            # after calibration finishes:
            if len(calib_frames) > 10:
                raw_mean    = np.mean(calib_frames, axis=0).reshape(1, -1)
                scaled_mean = scaler.transform(raw_mean)
                calib_prob  = model.predict_proba(scaled_mean)[0][GOOD_CLASS_IDX]
                # set threshold just below your personal good posture score
                BASE_THRESHOLD = 0.5

                # calibration only shifts slightly
                calib_shift = np.mean(calib_prob) - 0.7
                
                THRESHOLD = np.percentile(calib_prob, 40) 
                print(f"Calibration done. Your good posture score: {calib_prob:.2f} → threshold set to {THRESHOLD:.2f}")
            else:
                print("Calibration failed — no pose detected. Running without offset.")
        calib_mean = np.mean(calib_frames, axis=0)

        cv2.imshow('Posture Check', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    # ── INFERENCE PHASE ──────────────────────────────────────────────────────
    if result.pose_landmarks:
        landmarks   = result.pose_landmarks[0]
        feat        = compute_features(landmarks)

        # apply personal offset if calibrated
        feat_delta = feat - calib_mean
        feat_final = np.concatenate([feat, feat_delta])
        feat_scaled = scaler.transform(feat_final)
        prob_good = model.predict_proba(feat_scaled)[0][GOOD_CLASS_IDX]
        prob_buffer.append(prob_good)
        smoothed     = np.mean(prob_buffer)

        pred  = 0 if smoothed > THRESHOLD else 1
        label = "Good" if pred == 0 else "Bad"
        color = (0, 220, 80) if pred == 0 else (0, 0, 220)

        cv2.putText(frame, f"Posture: {label}", (10, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        cv2.putText(frame, f"Smooth: {smoothed:.2f}  inst: {prob_good:.2f}", (10, 85),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1, cv2.LINE_AA)

        h, w = frame.shape[:2]
        for idx in LANDMARK_INDICES:
            lm     = landmarks[idx]
            cx, cy = int(lm.x * w), int(lm.y * h)
            cv2.circle(frame, (cx, cy), 5, color, -1)
        for (i, j) in CONNECTIONS:
            lm1, lm2 = landmarks[i], landmarks[j]
            cv2.line(frame,
                     (int(lm1.x * w), int(lm1.y * h)),
                     (int(lm2.x * w), int(lm2.y * h)),
                     color, 2)
    else:
        cv2.putText(frame, "No pose detected", (10, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (80, 80, 200), 2, cv2.LINE_AA)

    cv2.imshow('Posture Check', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
detector.close()

CALIBRATION: sit in good posture for 15 seconds...
Calibration done. Your good posture score: 0.09 → threshold set to 0.09
